##### Copyright 2024 Google LLC。

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 微調Gemma以進行函數調用

歡迎閱讀有關函數呼叫fine-tuning 和 [Gemma](https://huggingface.co/google/gemma-2b) 的逐步指南。

[**Gemma**](https://ai.google.dev/gemma) 是 Google 推出的一系列輕量級、最先進的開放模型，採用與創建 Gemini 模型相同的研究和技術構建。它們是文字到文字、僅限解碼器的大型語言模型，提供英文版本，具有開放權重、預訓練變體和指令調整變體。 Gemma 模型非常適合各種文本生成任務，包括問答、摘要和推論。它們的尺寸相對較小，因此可以將它們部署在資源有限的環境中，例如筆記型電腦、桌上型電腦或您自己的雲端基礎設施，從而實現對最先進人工智慧模型的民主化訪問，並幫助促進每個人的創新。
**函數呼叫finetuning**是透過函數呼叫功能增強 LLM 效能的關鍵一步。它涉及在 prompt 的 dataset 和相應的函數呼叫上訓練模型，使其能夠準確識別給定任務的適當函數。透過fine-tuning模型，它學會更好地理解自然語言的細微差別，識別prompt背後的意圖，並選擇最合適的函數。
此 notebook 使用 [Torch XLA](https://github.com/pytorch/xla) 和 Hugging Face 的 [**變壓器強化學習 (TRL)**](https://github.com/huggingface/trl) framework 來呼叫 finetuning 函數。
[**Torch XLA**](https://pytorch.org/xla/) 讓您能夠利用 TPU（張量處理單元）的運算能力來高效訓練深度學習模型。透過將 PyTorch 與 [XLA（加速線性代數）](https://openxla.org/xla) 編譯器連接，Torch XLA 將 PyTorch 操作轉換為可以在 TPU 上執行的 XLA 操作。這意味著您可以像往常一樣在 PyTorch 中編寫模型，而 Torch XLA 會處理底層計算以在 TPU 上高效執行它們。
**Transformer 強化學習 (TRL)**](https://github.com/huggingface/trl) 是 Hugging Face 開發的 framework，使用監督微調 (SFT)、獎勵建模 (RM)、近端策略優化 (PPO)、直接偏好優化 (DPO) 等方法來微調和對齊擴散和對齊語言和對齊。
要了解有關如何使用 Torch XLA 和 TRL 微調 Gemma 的更多信息，請查看 [Gemma Cookbook](https://github.com/google-gemini/gemma-cookbook/blob/main/Gemma/Finetune_with_Torch_XLA.ipynb) 中的 **使用 Torch XLA** notebook 進行微調。
<table align="left"> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/Gemma/[Gemma_2]Finetune_with_Function_Calling.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>
<br><br>
[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)]("https://www.kaggle.com/notebooks/welcome?src=https://github.com/google-gemini/gemma-cookbook/blob/main/Gemma/[Gemma_2]Finetune_with_Function_Calling.ipynb")

## 設定

### 選擇執行時環境

首先，您可以選擇 **Google Colab** 或 **Kaggle** 作為您的平台。選擇一個，然後從那裡繼續。
- #### **Google Colab** <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d0/Google_Colaboratory_SVG_Logo.svg/1200px-Google_Colaboratory_SVG_Logo.svg.png" alt="Google Colab" width="30"/>

  1. 按一下「**在 Colab** 中開啟」。
  2. 在選單中，前往 **執行時間** > **變更 runtime 類型**。
  3. 在 **硬體加速器** 下，選擇 **TPU**。
  4. 確保 **TPU 類型** 設定為 **TPU v2-8**。

- #### **Kaggle** <img src="https://upload.wikimedia.org/wikipedia/commons/7/7c/Kaggle_logo.png" alt="Kaggle" width="40"/>

  1. 按一下「**在 Kaggle** 中開啟」。
  2. 點選右側邊欄中的**設定**。
  3. 在 **加速器** 下，選擇 **TPU**。
- 注意：Kaggle 目前提供 **TPU v3-8**。  4. 儲存設置，notebook 將在 TPU 支援下重新啟動。


### Gemma 使用Hugging Face

在深入學習本教學之前，讓我們先設定Gemma：
1. **建立一個 Hugging Face 帳戶**：如果您沒有帳戶，您可以[此處]註冊一個免費帳戶(https://huggingface.com/join)。
2. **造訪Gemma型號**：造訪[Gemma型號頁面](https://huggingface.com/collections/google/gemma-2-release-667d6600fd5220e7b967f315)並接受使用條件。
3. **產生Hugging Facetoken**：前往您的Hugging Face [設定頁面](https://huggingface.com/settings/tokens)並產生新的存取token（最好具有`write`權限）。在本教學的後面部分，您將需要這個token。

**完成這些步驟後，您就可以進入下一部分，在 Colab 環境中設定環境變數。 **

### 設定您的憑證

要存取私有模型和dataset，您需要登入Hugging Face（HF）生態系統。
- #### **Google Colab** <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d0/Google_Colaboratory_SVG_Logo.svg/1200px-Google_Colaboratory_SVG_Logo.svg.png" alt="Google Colab" width="30"/>
如果您使用 Colab，您可以使用 Colab Secrets manager 安全地儲存 Hugging Face token (`HF_TOKEN`)：  1. 開啟 Google Colab notebook 並點選左側面板中的 🔑 Secrets 標籤。 <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="The Secrets tab is found on the left panel." width=50%>
  2. **新增Hugging Facetoken**：
- 建立一個新的secret，其**名稱**為`HF_TOKEN`。 - 將token 金鑰複製/貼上到`HF_TOKEN` 的**值** 輸入框中。 - **切換**左側的按鈕以允許notebook訪問secret
- #### **Kaggle** <img src="https://upload.wikimedia.org/wikipedia/commons/7/7c/Kaggle_logo.png" alt="Kaggle" width="40"/>
要在此 notebook 中安全地使用 Hugging Face token (`HF_TOKEN`)，您需要將其作為 secret 添加到 Kaggle 環境中：  1. 開啟 Kaggle notebook 並找到 notebook 介面頂部的 **插件** 選單。
  2. 點選 **Secrets** 來管理您的環境secrets。
<img src="https://i.imgur.com/vxrtJuM.png" alt="The Secrets option is found at the top." width=50%>  3. **新增Hugging Facetoken**：
- 點選「**新增secret**」按鈕。 - 在**標籤**欄位中，輸入`HF_TOKEN`。 - 在 **值** 欄位中，貼上 Hugging Face token。 - 點選「**儲存**」新增secret。

此程式碼會擷取您的 secrets 並將它們設為環境變量，您將在本教學後面使用它們。

In [ ]:
import os
import sys

if 'google.colab' in sys.modules:
    # Running on Colab
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
elif os.path.exists('/kaggle/working'):
    # Running on Kaggle
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = user_secrets.get_secret("HF_TOKEN")
else:
    # Not running on Colab or Kaggle
    raise EnvironmentError('This notebook is designed to run on Google Colab or Kaggle.')

### 安裝依賴項

接下來，您將使用 Torch XLA 在 TPU VM 上安裝 fine-tuning 和 Gemma 模型所需的所有 Python 軟體包來設定環境。

In [ ]:
# Uninstalling any existing TensorFlow installations and then install the CPU-only version to avoid conflicts while using the TPU.
!pip uninstall -y tensorflow tf-keras
!pip install tensorflow==2.18.0 tf-keras==2.18.0

!pip uninstall tensorflow -y
!pip install tensorflow-cpu==2.18.0 -q

# Install the appropriate Hugging Face libraries to ensure compatibility with the Gemma model and PEFT.
!pip install transformers==4.46.1 -U -q
!pip install datasets==3.1.0 -U -q
!pip install trl==0.12.0 peft==0.13.2 -U -q
!pip install accelerate==0.34.0 -U -q

# Install PyTorch and Torch XLA with versions compatible with the TPU runtime, ensuring efficient TPU utilization.
!pip install -qq torch~=2.5.0 --index-url https://download.pytorch.org/whl/cpu
!pip install -qq torch_xla[tpu]~=2.5.0 -f https://storage.googleapis.com/libtpu-releases/index.html

# Install the `tpu-info` package to display TPU-related information
!pip install tpu-info

Found existing installation: tensorflow 2.15.0
Uninstalling tensorflow-2.15.0:
  Successfully uninstalled tensorflow-2.15.0
Found existing installation: tf_keras 2.15.1
Uninstalling tf_keras-2.15.1:
  Successfully uninstalled tf_keras-2.15.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.3/615.3 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 381.3/381.3 kB 18.7 MB/s eta 0:00:00
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.2.0
    Uninstalling ml-dtypes-0.2.0:
      Successfully uninstalled ml-dtypes-0.2.0
  Attempting uninstall: tensorboard
    Found

**注意**：確保您的 PyTorch 和 Torch XLA 版本與您正在使用的 TPU 相容。

### 驗證 TPU 設定

您執行 `!tpu-info` 來驗證 TPU 是否已正確初始化。

In [ ]:
!tpu-info

TPU Chips                                     
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━┓
┃ Chip        ┃ Type        ┃ Devices ┃ PID  ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━┩
│ /dev/accel0 │ TPU v2 chip │ 2       │ None │
│ /dev/accel1 │ TPU v2 chip │ 2       │ None │
│ /dev/accel2 │ TPU v2 chip │ 2       │ None │
│ /dev/accel3 │ TPU v2 chip │ 2       │ None │
└─────────────┴─────────────┴─────────┴──────┘
Libtpu metrics unavailable. Is there a framework using the TPU? See https://github.com/google/cloud-accelerator-diagnostics/tree/main/tpu_info for more information


如果一切設定正確，您應該會看到列印的 TPU 詳細資訊。

## 函數呼叫的微調 Gemma 2

### 正在初始化Gemma 2模型

您將透過從 HuggingFace 載入預先訓練的 Gemma 2 模型，從 `transformers` library 初始化 `AutoModelForCausalLM`。您也將使用`transformers` library 中的`AutoTokenizer` 來初始化所選模型(`google/gemma-2-2b-it`) 的tokenizer。

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

# Define model names
model_name = "google/gemma-2-2b-it"
new_model = "gemma-func-ft"

# Load the Gemma pre-trained model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16
)

# You must disable the cache to prevent issues during training
model.config.use_cache = False

# Load the Gemma tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# You adjust the tokenizer's padding side to ensure compatibility during TPU
# training.
tokenizer.padding_side = "right" # Fix overflow issue with bf16/fp16 training

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

啟用單程式多資料（SPMD）模式，
它允許跨多個 TPU 核心並行執行。

In [ ]:
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr

xr.use_spmd()

### 加載dataset

對於本指南，您將使用Hugging Face 中的現有dataset。如果您願意，可以將其替換為 dataset。
本指南選擇的 dataset 是 [**lilacai/glaive-function-calling-v2-sharegpt**](https://huggingface.co/datasets/lilacai/glaive-function-calling-v2-sharegpt)，它是 glaiveai 原始 **glaive-function-calling-v2** dataset 的 ShareGPT 版本。 glaive-function-calling-v2 dataset 是超過 113,000 個 prompt 和對應函數呼叫的集合，可以微調語言模型以準確識別給定任務的適當函數。
**學分：** **https://huggingface.co/lilacai**

In [ ]:
from datasets import Dataset, load_dataset

# Only the first 15% of the `train` split is used for training. A smaller
# subsection of the dataset is selected to avoid out-of-memory crashes.
dataset = load_dataset("lilacai/glaive-function-calling-v2-sharegpt", split="train[:15%]")

README.md:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

(…)-00000-of-00002-6f3344faa23e9b0a.parquet:   0%|          | 0.00/98.0M [00:00<?, ?B/s]

(…)-00001-of-00002-41f063cddf49c933.parquet:   0%|          | 0.00/98.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112960 [00:00<?, ? examples/s]

讓我們看幾個範例來了解數據。

In [ ]:
dataset[10]['conversations']

[{'from': 'system',
  'value': 'You are a helpful assistant with access to the following functions. Use them if required -\n{\n    "name": "calculate_discount",\n    "description": "Calculate the discount amount based on original price and discount percentage",\n    "parameters": {\n        "type": "object",\n        "properties": {\n            "original_price": {\n                "type": "number",\n                "description": "The original price of the item"\n            },\n            "discount_percentage": {\n                "type": "number",\n                "description": "The percentage discount"\n            }\n        },\n        "required": [\n            "original_price",\n            "discount_percentage"\n        ]\n    }\n}\n'},
 {'from': 'human',
  'value': "Hi, I saw a dress that I liked in a store. It was originally priced at $200 but it's on a 20% discount. Can you help me calculate how much I will save?"},
 {'from': 'gpt',
  'value': '<functioncall> {"name": "cal

### 建立自訂聊天模板

Hugging Face 支援聊天模板，可用於定義將對話轉換為單一 token 可化字串的結構和格式，這是語言模型期望的輸入格式。查看[聊天範本文件](https://huggingface.co/docs/transformers/main/en/chat_templating) 以了解有關範本以及如何建立自訂新範本的更多資訊。
由於Gemma不支援系統指令，因此您將提供系統輸入作為使用者輸入。要了解有關 Gemma 所需格式的更多信息，請查看 [Gemma 格式化文件](https://ai.google.dev/gemma/docs/formatting)。

In [ ]:
# Reference: https://github.com/unslothai/unsloth/blob/main/unsloth/chat_templates.py#L383

chat_template = \
    "{{ bos_token }}"\
    "{% if messages[0]['from'] == 'system' %}"\
        "{{'<start_of_turn>user\n' + messages[0]['value'] | trim + ' ' + messages[1]['value'] | trim + '<end_of_turn>\n'}}"\
        "{% set messages = messages[2:] %}"\
    "{% endif %}"\
    "{% for message in messages %}"\
        "{% if message['from'] == 'human' %}"\
            "{{'<start_of_turn>user\n' + message['value'] | trim + '<end_of_turn>\n'}}"\
        "{% elif message['from'] == 'gpt' %}"\
            "{{'<start_of_turn>model\n' + message['value'] | trim + '<end_of_turn>\n' }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}"\
        "{{ '<start_of_turn>model\n' }}"\
    "{% endif %}"

tokenizer.chat_template = chat_template

### 定義格式化函數

格式化函數將上面建立的範本套用到dataset中的每一行，並將其轉換為適合訓練的格式。

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False,
                      add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/16944 [00:00<?, ? examples/s]

### 清理dataset。

從dataset 中刪除不必要的tokens。

In [ ]:
import pandas as pd

df_train = pd.DataFrame(dataset)
df_train["text"] = df_train["text"].apply(
    lambda x: x.replace("<|endoftext|>", ""))

pd.set_option('display.max_colwidth', None)
print(df_train.head(1))

                                                                                                                                                                                                                                                                                                                                                                 chat  \
0  USER: Hi, I have a list of numbers and I need to find the median. The numbers are 5, 2, 9, 1, 7, 4, 6, 3, 8.\n\n\nASSISTANT: <functioncall> {"name": "calculate_median", "arguments": '{"numbers": [5, 2, 9, 1, 7, 4, 6, 3, 8]}'} <|endoftext|>\n\n\nFUNCTION RESPONSE: {"median": 5}\n\n\nASSISTANT: The median of your list of numbers is 5. <|endoftext|>\n\n\n   

                                                                                                                                                                                                                                                                                     

將dataset 轉換回Hugging Face 的`Dataset` 格式。

In [ ]:
dataset = Dataset.from_pandas(df_train[['text']])

dataset

Dataset({
    features: ['text'],
    num_rows: 16944
})

### LoRA設定

LoRA（低階適應）將小型可訓練矩陣引入模型架構中，專門針對 Transformer 模型的注意力層。 LoRA 沒有更新完整的權重矩陣，而是添加了秩分解矩陣，使適應更有效率。
在這裡，您設定以下參數：- `r`到16，它控制適應矩陣的等級。
- `lora_alpha` 至 16 進行縮放。
- `lora_dropout` 為 0，因為它已被最佳化。

In [ ]:
from peft import LoraConfig, PeftModel

# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=16,       # Alpha parameter for LoRA scaling
    lora_dropout=0,    # Dropout probability for LoRA layers
    r=16,                # LoRA attention dimension
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",]
)

**完全分片資料並行 (FSDP)** 設定在 `fsdp_config` 中設置，啟用 [**全模型分片**](https://pytorch.org/docs/stable/fsdp.html#torch.distributed.fsdp.ShardingStrategy) 和 [**梯度 checkpointing**](https://huggingface.co/docs/transformers/v4.19.4/en/performance#gradient-checkpointing) 以提高 TPU 上的內存效率，並指定應啟用梯度@P001

In [ ]:
# Set up the FSDP config. To enable FSDP via SPMD, set xla_fsdp_v2 to True.
fsdp_config = {
    "fsdp_transformer_layer_cls_to_wrap": [
        "Gemma2DecoderLayer"
    ],
    "xla": True,
    "xla_fsdp_v2": True,
    "xla_fsdp_grad_ckpt": True
}

### 設定訓練設定

設定定義如何訓練模型的訓練參數。
在這裡，您將定義以下參數：
- 對於培訓：
  - `output directory`
  - `max steps`
  - `batch sizes`

- 優化訓練過程：
  - `learning rate`
  - `optimizer`
  - `learning rate scheduler`

**注意：** `max_steps` 設定為 100 步以加快速度，但您可以設定`num_train_epochs=1` 進行完整執行。

In [ ]:
from trl import SFTTrainer, SFTConfig

# Set training parameters
training_arguments = SFTConfig(
    # ---Output settings--
    # Output directory where model predictions and checkpoints will be stored
    output_dir="./results",
    overwrite_output_dir=True,
    save_strategy="no",
    # ---Training settings---
    # Number of training epochs
    #num_train_epochs=1,
    # Number of training steps (overrides num_train_epochs)
    max_steps=100,
    # This is the global train batch size for SPMD
    # Batch size per GPU core for training
    per_device_train_batch_size=32,
    # Number of update steps to accumulate the gradients for
    gradient_accumulation_steps=1,
    # Optimizer to use
    optim="adafactor",
    # Required for SPMD
    dataloader_drop_last=True,
    fsdp="full_shard",
    fsdp_config=fsdp_config,
    # Initial learning rate (adafactor optimizer)
    learning_rate=0.0002,
    # Enable bfloat16 precision
    bf16=True,
    # Maximum gradient normal (gradient clipping)
    max_grad_norm=0.3,
    # Ratio of steps for a linear warmup (from 0 to learning rate)
    warmup_ratio=0.03,
    # Learning rate schedule (constant a bit better than cosine)
    lr_scheduler_type="linear",
    # Maximum sequence length to use
    max_seq_length=1024,
    dataset_text_field="text",
    dataset_kwargs={
        "add_special_tokens": False,
        "append_concat_token": False,
    },
    # Pack multiple short examples in the same input sequence
    # to increase efficiency
    packing=True,
    # ---Logging---
    # Log every X update step
    logging_steps=1,
    report_to="none",
    seed=42
)

### 訓練模型

[Huggingface 的 TRL](https://huggingface.co/docs/trl/index) 提供了一個用戶友好的 API，用於構建 SFT 模型並在您的 dataset 上訓練它們，只需幾行程式碼。這裡您將使用 Huggingface TRL 的 `SFTTrainer` 類別來訓練模型。該類繼承自Transformers library 中可用的`Trainer` 類，但專門針對受監督fine-tuning（指令調整）進行了優化。從 [官方 TRL SFT 文件](https://huggingface.co/docs/trl/sft_trainer) 中了解更多關於 SFFTrainer 的資訊。

In [ ]:
# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_arguments
)

Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:403: UserWarning: You passed a processing_class with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `processing_class.padding_side = 'right'` to your code.
  warnings.warn(
max_steps is given, it will override any value given in num_train_epochs
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:428: UserWarning: You passed `packing=True` to the SFTTrainer/SFTConfig, and you are training your model with `max_steps` strategy. The dataset will be iterated until the `max_steps` are reached.
  warnings.warn(


現在，讓我們透過呼叫`trainer.train()`來啟動fine-tuning進程，它使用`SFTTrainer`來處理訓練循環，包括資料載入、前向和後向傳遞以及優化器步驟，所有這些都根據您提供的設定進行設定。

In [ ]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/module.py:1810: UserWarning: For backward hooks to be called, module output should be a Tensor or a tuple of Tensors but received <class 'transformers.modeling_outputs.CausalLMOutputWithPast'>
  warnings.warn("For backward hooks to be called,"
/usr/local/lib/python3.10/dist-packages/torch_xla/utils/checkpoint.py:183: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  torch.cuda.amp.autocast(**ctx.gpu_autocast_kwargs), \
/usr/local/lib/python3.10/dist-packages/torch_xla/utils/checkpoint.py:184: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):


Step,Training Loss
1,2.093800
2,2.125000
3,2.078100
4,1.765600
5,1.546900
6,1.218800
7,1.164100
8,1.039100
9,1.125000
10,0.953100


/usr/local/lib/python3.10/dist-packages/torch_xla/core/xla_model.py:1457: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  xldata.append(torch.load(xbio))


TrainOutput(global_step=100, training_loss=0.708515625, metrics={'train_runtime': 849.5865, 'train_samples_per_second': 3.767, 'train_steps_per_second': 0.118, 'total_flos': 5.18083433201664e+16, 'train_loss': 0.708515625, 'epoch': 0.3861003861003861})

訓練完成後，儲存微調後的模型，透過`trainer.model.to('cpu')`將其移至 CPU 以確保相容性，然後呼叫`save_pretrained(new_model)`將模型權重和設定檔儲存到`new_model`（**gemma-func-ft**）指定的目錄中。這允許您稍後重新加載並使用微調後的模型進行 inference 或進一步培訓。

In [ ]:
# Remove the model weights directory if it exists
!rm -rf gemma-func-ft

# Save the LoRA adapter
trainer.model.to('cpu').save_pretrained(new_model)

## 提示使用新微調的模型


現在您終於微調了自訂 Gemma 模型，讓我們重新載入 LoRA 適配器權重以最終 prompt 使用它，並驗證它是否真正按預期工作。

為此，請使用以下步驟正確地重新加載適配器重量：
- 使用 `AutoModelForCausalLM.from_pretrained` 首先載入 **基本 Gemma 模型**，同時設定 `low_cpu_mem_usage=True` 以優化記憶體消耗（因為您使用的是 TPU），並設定 `torch_dtype=torch.bfloat16` 以與微調模型保持一致。

- 使用 `PeftModel.from_pretrained` 載入您先前儲存到基本模型中的 **微調 LoRA 適配器**，其中 `new_model` 是包含微調權重的目錄。

- `model.merge_and_unload` 函數**將**LoRA 適配器權重**與**基本模型權重**合併並卸載適配器，從而產生一個可用於inference 的獨立模型。

In [ ]:
# Reload the fine-tuned Gemma model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.bfloat16
)
model = PeftModel.from_pretrained(base_model, new_model)
model = model.merge_and_unload()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

重新載入 tokenizer 以確保其與模型設定匹配，並像以前一樣調整填充側。

In [ ]:
# Reload tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "right"

現在，使用範例 prompt 測試微調模型，首先使用 tokenizer 產生輸入 id，然後依靠重新載入的微調模型使用 `model.generate()` 產生回應。

In [ ]:
input_text = """\
<start_of_turn>user
You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "calculate_median",
    "description": "Calculate the median of a list of numbers",
    "parameters": {
        "type": "object",
        "properties": {
             "numbers": {
                 "type": "array",
                 "items": {
                     "type": "number"
                 },
                 "description": "The list of numbers"
             }
        }
        "required": [
            "numbers"
        ]
    }
}
To use these functions respond with:
<functioncall> {"name": "function_name", "arguments": {"arg_1": "value_1", "arg_1": "value_1", ...}} </functioncall>

Then finally respond with:
Answer:

<end_of_turn>
<start_of_turn>user
USER: Hi, I have a list of numbers and I need to find the median. The numbers are [5, 2, 9, 1, 7, 4, 6, 3, 8]
<end_of_turn>
<start_of_turn>model
<functioncall>
"""

In [ ]:
input_ids = tokenizer(input_text, return_tensors="pt").to("cpu")
outputs = model.generate(**input_ids, max_new_tokens = 512)

最後，您使用 `tokenizer.decode` 將輸出 tokens 解碼回人類可讀的文字並列印結果，以便您可以查看微調後的模型如何回應 prompt。

In [ ]:
print(tokenizer.decode(outputs[0]))

<bos><start_of_turn>user
You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "calculate_median",
    "description": "Calculate the median of a list of numbers",
    "parameters": {
        "type": "object",
        "properties": { 
             "numbers": {
                 "type": "array",
                 "items": {
                     "type": "number"              
                 },
                 "description": "The list of numbers"
             }      
        }       
        "required": [
            "numbers"       
        ]    
    }
}
To use these functions respond with:
<functioncall> {"name": "function_name", "arguments": {"arg_1": "value_1", "arg_1": "value_1", ...}} </functioncall>

Then finally respond with:
Answer:

<end_of_turn>
<start_of_turn>user
USER: Hi, I have a list of numbers and I need to find the median. The numbers are [5, 2, 9, 1, 7, 4, 6, 3, 8]
<end_of_turn>
<start_of_turn>model
<functioncall>
{"nam

恭喜！您已成功使用 Torch XLA 和 PEFT 与 TPU 上的 LoRA 微调 Gemma 进行函数调用。至此，您已经了解了从设置环境到训练和测试模型的整个过程。

## 接下來怎麼辦？
您的後續步驟可能包括以下內容：
- **使用不同的 dataset 進行實驗**：嘗試在其他函數上使用 fine-tuning 呼叫 [Hugging Face](https://huggingface.co/docs/datasets/en/index) 中的 datasets 或您自己的資料。

- **調整超參數**：調整訓練參數（例如，學習率、批量大小、時期、LoRA 設定）以優化效能和
提高培訓效率。
- **嘗試不同的模板**：嘗試不同的聊天模板並嘗試提高效能。

透過探索這些活動，您將加深理解並進一步增強經過微調的 Gemma 模型。快樂實驗！